# Fine-tune Llama 3.1 8B for Support Email Extraction

**Goal:** Prove/disprove that fine-tuned Llama 3.1 8B beats Llama 3.3 70B on urgency classification at 5x lower inference cost.

**Runtime:** ~45 min on free T4. Run cells top-to-bottom.

**Works in:** Google Colab (primary) and local Jupyter with a GPU.

**Sections:**
0. Install & environment setup  
1. Load Llama 3.1 8B (4-bit)  
2. Baseline eval (no fine-tuning)  
3. Apply LoRA  
4. Train (3 epochs)  
5. Post-training eval + comparison  
6. Save adapter

## Section 0 — Install & Environment Setup

In [ ]:
import sys, os

# Detect Colab early (needed before the full cell-env block runs)
try:
    import google.colab
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    # T4 on Colab: [colab-new] selects the correct pre-built CUDA wheel automatically.
    # trl pinned to 0.8.6 — newer versions rename SFTConfig params and break dataset_text_field.
    os.system('pip install "unsloth[colab-new]" "trl==0.8.6" accelerate bitsandbytes xformers -q')
else:
    # Local GPU (e.g. RTX, A-series): plain unsloth without the Colab-specific extra.
    # Requires a CUDA-capable GPU — model load will raise NotImplementedError on CPU-only machines.
    os.system('pip install unsloth "trl==0.8.6" accelerate bitsandbytes -q')

print("Install done.")

In [ ]:
import os
import sys

# Detect runtime environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in {'Google Colab' if IN_COLAB else 'local Jupyter'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR   = "/content/data"
    OUTPUT_DIR = "/content"
    SAVE_PATH  = "/content/drive/MyDrive/llama31-8b-support-lora"
else:
    # Walk up from cwd until we find the directory that contains data/train.jsonl.
    # Works regardless of where Jupyter / VS Code launched the kernel from.
    def _find_repo():
        cwd = os.getcwd()
        for candidate in [cwd,
                          os.path.dirname(cwd),
                          os.path.dirname(os.path.dirname(cwd))]:
            candidate = os.path.abspath(candidate)
            if os.path.exists(os.path.join(candidate, "data", "train.jsonl")):
                return candidate
        return cwd  # fallback — cell-upload will report any missing files

    _repo      = _find_repo()
    DATA_DIR   = os.path.join(_repo, "data")
    OUTPUT_DIR = os.path.join(_repo, "results")
    SAVE_PATH  = os.path.join(_repo, "results", "lora_adapter")

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SAVE_PATH,  exist_ok=True)

print(f"DATA_DIR:   {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"SAVE_PATH:  {SAVE_PATH}")

In [ ]:
import json

if IN_COLAB:
    from google.colab import files
    print("Upload: train.jsonl, eval.jsonl, baseline_together_70b.json")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = os.path.join(DATA_DIR, fname)
        with open(dest, "wb") as f:
            f.write(data)
        print(f"  Saved {fname} ({len(data):,} bytes)")
else:
    print("Local mode: reading data files from", DATA_DIR)
    for fname in ["train.jsonl", "eval.jsonl", "baseline_together_70b.json"]:
        src = os.path.join(DATA_DIR, fname)
        # baseline_together_70b.json lives in results/, not data/
        if not os.path.exists(src) and fname == "baseline_together_70b.json":
            src = os.path.join(OUTPUT_DIR, fname)
        print(f"  {'OK' if os.path.exists(src) else 'MISSING'}: {src}")

TRAIN_FILE  = os.path.join(DATA_DIR, "train.jsonl")
EVAL_FILE   = os.path.join(DATA_DIR, "eval.jsonl")
_b70 = os.path.join(DATA_DIR, "baseline_together_70b.json")
BASELINE_70B_FILE = _b70 if os.path.exists(_b70) else os.path.join(OUTPUT_DIR, "baseline_together_70b.json")

train_examples = [json.loads(l) for l in open(TRAIN_FILE, encoding="utf-8") if l.strip()]
eval_examples  = [json.loads(l) for l in open(EVAL_FILE,  encoding="utf-8") if l.strip()]
print(f"\nTrain: {len(train_examples)} examples")
print(f"Eval:  {len(eval_examples)} examples")

# Hash-check: zero overlap between train emails and eval emails
train_emails = {ex["messages"][1]["content"] for ex in train_examples}
eval_emails  = {ex["email"] for ex in eval_examples}
overlap = train_emails & eval_emails
assert len(overlap) == 0, f"DATA LEAK: {len(overlap)} emails appear in both train and eval!"
print("Hash-check passed: zero overlap between train and eval.")

## Section 1 — Load Llama 3.1 8B Instruct (4-bit QLoRA)

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    print("GPU:", result.stdout.strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU: nvidia-smi not found — no NVIDIA GPU detected (expected locally; Colab has T4)")

In [ ]:
MAX_SEQ_LENGTH = 512

try:
    from unsloth import FastLanguageModel
    import torch
    GPU_AVAILABLE = True
except (ImportError, NotImplementedError, Exception) as _e:
    GPU_AVAILABLE = False
    FastLanguageModel = None
    print("=" * 60)
    print("NO GPU / Unsloth unavailable — model cells will be skipped.")
    print(f"Reason: {_e}")
    print()
    print("To run the full notebook:")
    print("  1. Google Colab (free T4): Runtime → Change runtime type → T4 GPU")
    print("  2. Local NVIDIA GPU (RTX 12 GB+): install CUDA + unsloth")
    print("=" * 60)

if GPU_AVAILABLE:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct",
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    print(f"Loaded. dtype={model.dtype}  device={next(model.parameters()).device}")
else:
    model = tokenizer = None
    print("Skipped model load (no GPU).")

## Section 2 — Baseline Evaluation (Before Fine-Tuning)

We define the evaluation function here and **reuse it** after training.  
Logic mirrors `scripts/evaluate_together.py` exactly so results are comparable.

In [ ]:
from collections import defaultdict

SYSTEM_PROMPT = (
    "You extract structured data from customer support emails. "
    "Return only a single valid JSON object with fields: "
    "customer_name (string or null), product (string), "
    "issue_category (one of: billing, technical, account, feature_request, other), "
    "urgency (one of: low, medium, high, critical), "
    "summary (one short sentence). No extra text."
)
FIELDS = ["customer_name", "product", "issue_category", "urgency", "summary"]


def safe_json_parse(s):
    """Mirror of evaluate_together.py safe_json_parse."""
    s = (s or "").strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[1] if "\n" in s else s
        s = s.rsplit("```", 1)[0].strip()
        if s.startswith("json"):
            s = s[4:].strip()
    start = s.find("{")
    end   = s.rfind("}")
    if start != -1 and end > start:
        s = s[start:end+1]
    try:
        return json.loads(s), True
    except json.JSONDecodeError:
        return None, False


def field_match(predicted, expected, field):
    """Mirror of evaluate_together.py field_match."""
    if field == "summary":
        if not isinstance(predicted, str) or not isinstance(expected, str):
            return False
        p = {w.lower().strip(".,!?") for w in predicted.split() if len(w) > 3}
        e = {w.lower().strip(".,!?") for w in expected.split()   if len(w) > 3}
        if not p or not e:
            return False
        return len(p & e) / max(len(e), 1) >= 0.4
    if field == "customer_name":
        if predicted is None and expected is None:
            return True
        if predicted is None or expected is None:
            return False
        return expected.split()[0].lower() in predicted.lower() if predicted else False
    if predicted is None:
        return False
    return str(predicted).strip().lower() == str(expected).strip().lower()


def run_inference(model, tokenizer, email_text, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": email_text},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,   # appends <|start_header_id|>assistant<|end_header_id|>
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,             # greedy — deterministic eval
            pad_token_id=tokenizer.eos_token_id,
        )

    # Slice only the newly generated tokens (not the prompt)
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def evaluate_model(model, tokenizer, eval_path, label):
    FastLanguageModel.for_inference(model)  # enable Unsloth 2x-faster inference kernel

    examples = [json.loads(l) for l in open(eval_path, encoding="utf-8") if l.strip()]
    print(f"\n{'='*60}")
    print(f"Evaluating: {label}")
    print(f"Examples:   {len(examples)}")
    print(f"{'='*60}")

    results = []
    json_valid = exact_match = 0
    field_correct = defaultdict(int)

    for i, ex in enumerate(examples, 1):
        email, expected = ex["email"], ex["expected"]
        raw = run_inference(model, tokenizer, email)
        predicted, valid = safe_json_parse(raw)

        if valid:
            json_valid += 1

        per_field = {}
        all_match = True
        for f in FIELDS:
            ok = valid and field_match(
                predicted.get(f) if predicted else None,
                expected[f], f
            )
            per_field[f] = ok
            if ok:
                field_correct[f] += 1
            else:
                all_match = False

        is_exact = valid and all_match
        if is_exact:
            exact_match += 1

        mark = "✓" if is_exact else "✗"
        urg_pred = predicted.get("urgency", "?") if predicted else "?"
        print(f"  [{i:2d}/{len(examples)}] {mark}  "
              f"urgency={expected['urgency']:8s} pred={urg_pred}")

        results.append({
            "i": i, "email": email, "expected": expected,
            "raw_output": raw, "predicted": predicted,
            "valid_json": valid, "exact_match": is_exact,
            "field_match": per_field,
        })

    n = len(examples)
    metrics = {
        "label": label,
        "n_examples": n,
        "json_valid_rate":  round(json_valid  / n, 4),
        "exact_match_rate": round(exact_match / n, 4),
        "field_accuracy": {f: round(field_correct[f] / n, 4) for f in FIELDS},
        "details": results,
    }

    print(f"\n--- {label} ---")
    print(f"  JSON valid:   {metrics['json_valid_rate']*100:.1f}%")
    print(f"  Exact match:  {metrics['exact_match_rate']*100:.1f}%")
    for f in FIELDS:
        print(f"  {f:<22} {metrics['field_accuracy'][f]*100:.1f}%")

    return metrics, results

print("Evaluation functions defined.")

In [ ]:
if not GPU_AVAILABLE:
    print("Skipped (no GPU). Run in Colab to get baseline metrics.")
else:
    # ~4-5 minutes on T4
    base_metrics, base_results = evaluate_model(
        model, tokenizer,
        eval_path=EVAL_FILE,
        label="Llama-3.1-8B-Instruct (base, no fine-tuning)",
    )

    base_out = os.path.join(OUTPUT_DIR, "baseline_llama31_8b.json")
    with open(base_out, "w", encoding="utf-8") as f:
        summary = {k: v for k, v in base_metrics.items() if k != "details"}
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {base_out}")

## Section 3 — Apply LoRA Adapter

**Do not** call `FastLanguageModel.for_inference()` before training — it disables gradients.

In [ ]:
if not GPU_AVAILABLE:
    print("Skipped (no GPU).")
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
        loftq_config=None,
    )
    model.print_trainable_parameters()

## Section 4 — Format Data & Train

In [ ]:
from datasets import Dataset

raw_train = [json.loads(l) for l in open(TRAIN_FILE, encoding="utf-8") if l.strip()]
print(f"Training examples: {len(raw_train)}")

if not GPU_AVAILABLE:
    dataset = None
    print("Skipped dataset formatting (no tokenizer).")
else:
    def format_chat(ex):
        text = tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}

    dataset = Dataset.from_list([format_chat(ex) for ex in raw_train])
    print(f"Dataset columns: {dataset.column_names}")
    print(f"\nSample (first 300 chars):\n{dataset[0]['text'][:300]}")

In [ ]:
if not GPU_AVAILABLE:
    print("Skipped (no GPU). Run in Colab to train.")
else:
    from trl import SFTTrainer
    from transformers import TrainingArguments

    ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoints")

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            num_train_epochs=3,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=42,
            output_dir=ckpt_dir,
            report_to="none",
        ),
    )

    trainer_stats = trainer.train()
    print(f"\nTraining complete!")
    print(f"  Runtime:      {trainer_stats.metrics['train_runtime']:.0f}s")
    print(f"  Samples/sec:  {trainer_stats.metrics['train_samples_per_second']:.1f}")
    print(f"  Final loss:   {trainer_stats.metrics['train_loss']:.4f}")

## Section 5 — Post-Training Evaluation & Comparison

In [ ]:
if not GPU_AVAILABLE:
    print("Skipped (no GPU). Run in Colab to get fine-tuned metrics.")
else:
    # ~4-5 minutes on T4
    ft_metrics, ft_results = evaluate_model(
        model, tokenizer,
        eval_path=EVAL_FILE,
        label="Llama-3.1-8B-Instruct (fine-tuned QLoRA r=16)",
    )

    ft_out = os.path.join(OUTPUT_DIR, "finetuned_8b_metrics.json")
    with open(ft_out, "w", encoding="utf-8") as f:
        summary = {k: v for k, v in ft_metrics.items() if k != "details"}
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"Saved {ft_out}")

In [ ]:
if not GPU_AVAILABLE:
    print("Skipped (no GPU) — base_metrics and ft_metrics not available yet.")
    print("After running in Colab, load the saved JSONs to reproduce this table:")
    print(f"  base_metrics: {os.path.join(OUTPUT_DIR, 'baseline_llama31_8b.json')}")
    print(f"  ft_metrics:   {os.path.join(OUTPUT_DIR, 'finetuned_8b_metrics.json')}")
else:
    # Load 70B baseline
    with open(BASELINE_70B_FILE, encoding="utf-8") as f:
        data_70b = json.load(f)

    baseline_70b = {
        "label": "Llama-3.3-70B (no fine-tuning)",
        "json_valid_rate":  data_70b["json_valid_rate"],
        "exact_match_rate": data_70b["exact_match_rate"],
        "field_accuracy":   data_70b["field_accuracy"],
    }

    all_models = [base_metrics, ft_metrics, baseline_70b]

    col = 34
    labels = [m["label"][:col-2] for m in all_models]
    print(f"{'Metric':<24}" + "".join(f"{l:<{col}}" for l in labels))
    print("-" * (24 + col * len(all_models)))

    rows = [
        ("JSON Valid",         "json_valid_rate"),
        ("Exact Match",        "exact_match_rate"),
        ("  customer_name",    "customer_name"),
        ("  product",          "product"),
        ("  issue_category",   "issue_category"),
        ("  urgency",          "urgency"),
        ("  summary",          "summary"),
    ]

    for label, key in rows:
        print(f"{label:<24}", end="")
        for m in all_models:
            if key in ("json_valid_rate", "exact_match_rate"):
                val = m.get(key, 0)
            else:
                val = m["field_accuracy"].get(key, 0)
            print(f"{val*100:.1f}%".ljust(col), end="")
        print()

    print()
    urg_base = base_metrics["field_accuracy"]["urgency"]
    urg_ft   = ft_metrics["field_accuracy"]["urgency"]
    urg_70b  = baseline_70b["field_accuracy"]["urgency"]
    print(f"Urgency lift from fine-tuning:     {(urg_ft - urg_base)*100:+.1f}pp")
    print(f"Fine-tuned 8B beats 70B on urgency: "
          f"{urg_ft > urg_70b}  ({urg_ft*100:.1f}% vs {urg_70b*100:.1f}%)")

## Section 6 — Save LoRA Adapter

Saves **only the adapter** (~80 MB), not the merged model (~16 GB).  
In Colab, also downloads the two metrics JSON files locally.  
To reload later: `PeftModel.from_pretrained(base_model, SAVE_PATH)`

In [ ]:
if not GPU_AVAILABLE:
    print("Skipped (no GPU) — nothing to save.")
else:
    import shutil

    model.save_pretrained(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)

    shutil.copy(base_out, os.path.join(SAVE_PATH, "baseline_llama31_8b.json"))
    shutil.copy(ft_out,   os.path.join(SAVE_PATH, "finetuned_8b_metrics.json"))

    print(f"Saved adapter to {SAVE_PATH}:")
    for fn in sorted(os.listdir(SAVE_PATH)):
        size = os.path.getsize(os.path.join(SAVE_PATH, fn))
        print(f"  {fn:<42} {size/1e6:.1f} MB")

    if IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(base_out)
        colab_files.download(ft_out)
        print("Downloads triggered — check your browser.")
    else:
        print(f"Results saved to {OUTPUT_DIR}")